# Notebook 3 — Inventory EDA

**Goal:** Understand inventory levels, stockout patterns, and days-of-supply across stores and products.

Key questions:
1. What is the overall stockout rate? Is it realistic (2–5%)?
2. Which SKUs stockout most frequently?
3. Which stores have the worst inventory health?
4. Do stockouts concentrate around promotions and holidays?

This EDA directly feeds into Project 2 (Why Did We Run Out?).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sqlalchemy import create_engine

DB_URL = os.getenv('SUPPLY_CHAIN_DB_URL', '')
engine = create_engine(DB_URL)
SCHEMA = 'supply_chain'

def q(sql): return pd.read_sql(sql, engine)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Overall stockout rate

The industry benchmark for grocery retail is approximately 7–8% at the shelf level. Our simulation targets 2–5% (slightly better than average). This is the first number any inventory analyst will ask about.

In [ ]:
overall = q(f"""
SELECT
    COUNT(*) AS total_store_product_days,
    SUM(CASE WHEN is_stockout THEN 1 ELSE 0 END) AS stockout_days,
    ROUND(100.0 * SUM(CASE WHEN is_stockout THEN 1 ELSE 0 END) / COUNT(*), 2) AS stockout_rate_pct,
    ROUND(AVG(units_on_hand), 1) AS avg_units_on_hand,
    ROUND(AVG(days_of_supply), 2) AS avg_days_of_supply
FROM {SCHEMA}.inventory_snapshots
WHERE location_type = 'Store'
""")
print(overall.T)

## 2. Stockout rate by category

In [ ]:
by_cat = q(f"""
SELECT
    p.category,
    COUNT(*) AS total_days,
    SUM(CASE WHEN i.is_stockout THEN 1 ELSE 0 END) AS stockout_days,
    ROUND(100.0 * SUM(CASE WHEN i.is_stockout THEN 1 ELSE 0 END) / COUNT(*), 2) AS stockout_rate_pct,
    ROUND(AVG(i.days_of_supply), 2) AS avg_dos
FROM {SCHEMA}.inventory_snapshots i
JOIN {SCHEMA}.products p ON p.product_id = i.product_id
WHERE i.location_type = 'Store'
GROUP BY p.category
ORDER BY stockout_rate_pct DESC
""")
print(by_cat.to_string(index=False))

fig, ax = plt.subplots()
ax.barh(by_cat['category'], by_cat['stockout_rate_pct'], color='#EF5350')
ax.axvline(3.0, color='gray', linestyle='--', label='3% target')
ax.set_title('Stockout Rate by Category (%)')
ax.set_xlabel('Stockout Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Inventory health snapshot — using the analytical view

In [ ]:
# Distribution of inventory status right now (latest snapshot)
health = q(f"""
SELECT inventory_status, COUNT(*) AS n
FROM {SCHEMA}.v_inventory_health
GROUP BY inventory_status
ORDER BY n DESC
""")
print(health.to_string(index=False))

fig, ax = plt.subplots()
colors = {'Stockout': '#D32F2F', 'Below Safety Stock': '#FF7043',
          'Reorder Now': '#FFA726', 'Critical': '#FFEE58', 'Healthy': '#66BB6A'}
ax.bar(health['inventory_status'], health['n'],
       color=[colors.get(s, 'steelblue') for s in health['inventory_status']])
ax.set_title('Current Inventory Health Distribution')
ax.set_xlabel('Status')
ax.set_ylabel('# Store-Product Pairs')
plt.tight_layout()
plt.show()

## 4. Longest stockout episodes

Using the `v_stockout_calendar` view which identifies consecutive stockout days. Long stockout episodes are a red flag — they suggest either:
- Systematic under-ordering
- Supplier failure
- DC shortage
- Demand spike not anticipated

In [ ]:
long_stockouts = q(f"""
SELECT store_name, sku, product_name, category,
       stockout_start, stockout_end, stockout_days
FROM {SCHEMA}.v_stockout_calendar
ORDER BY stockout_days DESC
LIMIT 15
""")
print(long_stockouts.to_string(index=False))

## 5. Days-of-supply distribution

Days-of-supply (DOS) = units_on_hand / avg_daily_demand.

A DOS of 0 = stockout. A DOS > 30 = overstock (especially bad for perishables).
Healthy grocery is typically 7–14 days. Non-perishables: 14–30 days.

In [ ]:
dos_dist = q(f"""
SELECT
    p.category,
    p.is_perishable,
    ROUND(AVG(i.days_of_supply), 1) AS avg_dos,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY i.days_of_supply) AS p25_dos,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY i.days_of_supply) AS p50_dos,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY i.days_of_supply) AS p75_dos,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY i.days_of_supply) AS p95_dos
FROM {SCHEMA}.inventory_snapshots i
JOIN {SCHEMA}.products p ON p.product_id = i.product_id
WHERE i.location_type = 'Store' AND i.days_of_supply > 0
GROUP BY p.category, p.is_perishable
ORDER BY p.category, p.is_perishable
""")
print('Days-of-Supply Percentiles by Category:')
print(dos_dist.round(1).to_string(index=False))

## 6. Stockouts around holidays and promotions

**Interview insight:** Stockouts should spike around promotions (demand exceeds forecast) and holidays (unexpected demand surge). If they don't in our data, the simulation has a problem.

In [ ]:
stockout_timing = q(f"""
SELECT
    c.is_holiday,
    c.is_weekend,
    ROUND(100.0 * SUM(CASE WHEN i.is_stockout THEN 1 ELSE 0 END) / COUNT(*), 3) AS stockout_rate_pct
FROM {SCHEMA}.inventory_snapshots i
JOIN {SCHEMA}.calendar c ON c.date_id = i.snapshot_date
WHERE i.location_type = 'Store'
GROUP BY c.is_holiday, c.is_weekend
ORDER BY c.is_holiday DESC, c.is_weekend DESC
""")
print('Stockout rate by day type:')
print(stockout_timing)

## 7. Worst-performing stores (highest stockout rate)

This surfaces stores with systemic inventory problems — a key input for store operations teams.

In [ ]:
worst_stores = q(f"""
SELECT
    s.store_name,
    s.store_format,
    s.state,
    d.region,
    COUNT(*) AS total_days,
    ROUND(100.0 * SUM(CASE WHEN i.is_stockout THEN 1 ELSE 0 END) / COUNT(*), 2) AS stockout_rate_pct
FROM {SCHEMA}.inventory_snapshots i
JOIN {SCHEMA}.stores s ON s.store_id = i.store_id
JOIN {SCHEMA}.distribution_centers d ON d.dc_id = s.dc_id
WHERE i.location_type = 'Store'
GROUP BY s.store_name, s.store_format, s.state, d.region
ORDER BY stockout_rate_pct DESC
LIMIT 10
""")
print('Worst 10 stores by stockout rate:')
print(worst_stores.to_string(index=False))